# 0. Setup: Data Access, Panel Construction, and Policy Catalog

This notebook is the first reproducible step for the paper workflow. It starts from a local copy of the original iPinYou archive, builds the season-two opportunity panel, records archive metadata, and freezes the reserve/floor policy catalog used by the downstream experiment notebooks.

## What This Notebook Creates

- `artifacts/workspace/metadata/ipinyou_archive_inventory.csv`
- `artifacts/workspace/metadata/season2_file_matrix.csv`
- `artifacts/workspace/metadata/season2_panel_manifest.csv`
- `artifacts/workspace/metadata/season2_outcome_density.csv`
- `artifacts/workspace/data/processed/season2_bid_opportunity_panel/*.parquet`
- `artifacts/workspace/data/processed/season2_development_or_full_panel_current_scope.parquet`
- `artifacts/workspace/metadata/season3_panel_manifest.csv`
- `artifacts/workspace/metadata/season3_outcome_density.csv`
- `artifacts/workspace/data/processed/season3_bid_opportunity_panel/*.parquet`
- `artifacts/workspace/data/processed/season3_holdout_panel_current_scope.parquet`
- `artifacts/workspace/metadata/reserve_policy_price_landscape.csv`
- `artifacts/workspace/metadata/reserve_policy_registry.csv`

By default the notebook runs in quick mode. Set `FULL_RUN = True` below when you want the paper-scale setup.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root containing src/ and pyproject.toml")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print(f"Repository root: {REPO_ROOT}")

Repository root: /home/apex/Documents/ranking_sys/notebooks/writing/ads_marketplace_auction_experimentation/marketplace-policies-code


In [2]:
import pandas as pd

from config import RawPipelineConfig
from data_access import IpinYouArchive, Workspace
from panel_builder import OpportunityPanelBuilder
from policy_catalog import ReservePolicyCatalog
from progress import ProgressLogger

## Run Configuration

The original iPinYou archive is expected at `data/ipinyou/archive.zip`. Quick mode uses a bounded number of days and rows; full mode uses all available rows in the season-two development window.

In [3]:
FULL_RUN = True

config = RawPipelineConfig(
    data_root=REPO_ROOT / "data",
    workspace_root=REPO_ROOT / "artifacts" / "workspace",
    full_run=FULL_RUN,
    clean_workspace=True,
)
workspace = Workspace(config.workspace_root.expanduser().resolve())
progress = ProgressLogger(enabled=True)

print(f"Run mode: {'full' if config.full_run else 'quick'}")
print(f"Expected archive: {config.archive_path}")
print(f"Workspace: {workspace.root}")

Run mode: full
Expected archive: /home/apex/Documents/ranking_sys/notebooks/writing/ads_marketplace_auction_experimentation/marketplace-policies-code/data/ipinyou/archive.zip
Workspace: /home/apex/Documents/ranking_sys/notebooks/writing/ads_marketplace_auction_experimentation/marketplace-policies-code/artifacts/workspace


In [4]:
workspace.prepare(clean=config.clean_workspace)
archive = IpinYouArchive(config.archive_path)
builder = OpportunityPanelBuilder(archive, workspace, config, progress=progress)

inventory = builder.write_inventory()
inventory_summary = (
    inventory.groupby(["season", "file_kind"], as_index=False)
    .agg(files=("member", "size"), compressed_mb=("compressed_size", lambda s: round(s.sum() / 1_000_000, 2)))
    .sort_values(["season", "file_kind"])
)
inventory_summary

[     3.1s] Archived file inventory found 89 members.


,season,file_kind,files,compressed_mb
0,training1st,bid,7,858.38
1,training1st,clk,7,0.62
2,training1st,conv,7,0.01
3,training1st,imp,7,665.40
4,training2nd,bid,7,2643.86
5,training2nd,clk,7,0.73
6,training2nd,conv,7,0.05
7,training2nd,imp,7,794.14
8,training3rd,bid,10,652.96
9,training3rd,clk,9,0.20


## Build the Season-Two Opportunity Panel

This step materializes the opportunity-level panel used by the paper experiments. In quick mode, it only reads a bounded subset. In full mode, it reads all configured season-two dates and can take substantially longer.

In [5]:
panel = builder.build_season2()

panel_summary = pd.DataFrame(
    [
        {
            "rows": len(panel),
            "dates": panel["event_date"].nunique(),
            "filled_impressions": int(panel["filled"].sum()),
            "clicks": int(panel["clicked"].sum()),
            "conversions": int(panel["converted"].sum()),
            "fill_rate": panel["filled"].mean(),
            "click_rate_per_opportunity": panel["clicked"].mean(),
            "conversion_rate_per_opportunity": panel["converted"].mean(),
        }
    ]
)
panel_summary

[    29.6s] Starting: season-two bid-opportunity panel construction
[    29.6s] Building training2nd panel for date 20130606 (all rows).
[   273.9s] Built training2nd date 20130606: 9,586,949 opportunities, 1,815,075 fills, 1,149 clicks in 244.2s.
[   273.9s] Building training2nd panel for date 20130607 (all rows).
[   543.0s] Built training2nd date 20130607: 11,132,555 opportunities, 1,800,104 fills, 1,040 clicks in 269.12s.
[   543.0s] Building training2nd panel for date 20130608 (all rows).
[   686.3s] Built training2nd date 20130608: 5,226,937 opportunities, 1,629,307 fills, 1,154 clicks in 143.29s.
[   686.3s] Building training2nd panel for date 20130609 (all rows).
[   984.2s] Built training2nd date 20130609: 11,880,893 opportunities, 1,645,556 fills, 1,137 clicks in 297.88s.
[   984.2s] Building training2nd panel for date 20130610 (all rows).
[  1137.3s] Built training2nd date 20130610: 5,610,729 opportunities, 1,911,385 fills, 1,511 clicks in 153.08s.
[  1137.3s] Building train

,rows,dates,filled_impressions,clicks,conversions,fill_rate,click_rate_per_opportunity,conversion_rate_per_opportunity
0,53289330,7,12190344,8729,391,0.228758,0.000164,0.000007


In [7]:
panel.shape

(53289330, 43)

## Freeze the Reserve/Floor Policy Catalog

The policy catalog is parameterized by positive logged-floor quantiles from the setup panel. Downstream notebooks use this same frozen catalog so replay, guardrails, OPE, validation, and decision comparisons all refer to the same policy set.

In [8]:
positive_floors = panel.loc[panel["slot_floor_price"].gt(0), "slot_floor_price"]
quantile_series = positive_floors.quantile([0.25, 0.50, 0.75])
price_quantiles = {
    "q25": float(quantile_series.loc[0.25]),
    "q50": float(quantile_series.loc[0.50]),
    "q75": float(quantile_series.loc[0.75]),
}

price_summary = pd.DataFrame(
    [
        {"stat": "positive_floor_q25", "value": price_quantiles["q25"]},
        {"stat": "positive_floor_q50", "value": price_quantiles["q50"]},
        {"stat": "positive_floor_q75", "value": price_quantiles["q75"]},
        {"stat": "zero_floor_share", "value": panel["slot_floor_price"].fillna(0).eq(0).mean()},
        {"stat": "mean_bid_price", "value": panel["bid_price"].mean()},
    ]
)
price_summary.to_csv(workspace.metadata_dir / "reserve_policy_price_landscape.csv", index=False)

catalog = ReservePolicyCatalog(price_quantiles)
policy_registry = catalog.registry()
policy_registry.to_csv(workspace.metadata_dir / "reserve_policy_registry.csv", index=False)
policy_registry.to_csv(workspace.table_dir / "06_reserve_policy_registry.csv", index=False)

price_summary

,stat,value
0,positive_floor_q25,10.000000
1,positive_floor_q50,50.000000
2,positive_floor_q75,100.000000
3,zero_floor_share,0.114831
4,mean_bid_price,272.918178


In [9]:
policy_registry[["policy_number", "policy_id", "policy_family", "policy_label", "floor_rule"]]

,policy_number,policy_id,policy_family,policy_label,floor_rule
0,P0,logged_floor_status_quo,baseline,Logged Floor Status Quo,logged
1,P1,uniform_raise_05pct,uniform_percent,Uniform Raise 05Pct,logged * 1.05
2,P2,uniform_raise_10pct,uniform_percent,Uniform Raise 10Pct,logged * 1.10
3,P3,uniform_raise_15pct,uniform_percent,Uniform Raise 15Pct,logged * 1.15
4,P4,uniform_raise_20pct,uniform_percent,Uniform Raise 20Pct,logged * 1.20
5,P5,uniform_raise_30pct,uniform_percent,Uniform Raise 30Pct,logged * 1.30
6,P6,add_5_all_floors,absolute_increment,Add 5 All Floors,logged + 5
7,P7,add_10_all_floors,absolute_increment,Add 10 All Floors,logged + 10
8,P8,add_20_all_floors,absolute_increment,Add 20 All Floors,logged + 20
9,P9,min_positive_floor_q25,minimum_positive_floor,Min Positive Floor Q25,"max(logged, 10.00) if logged > 0"


## Build the Season-Three Holdout Panel

This step materializes the season-three holdout panel used for out-of-time validation. It writes per-day parquet shards, a combined holdout parquet file, a manifest, and daily outcome density.

In [10]:
season3_panel = builder.build_season3()

season3_summary = pd.DataFrame(
    [
        {
            "rows": len(season3_panel),
            "dates": season3_panel["event_date"].nunique(),
            "filled_impressions": int(season3_panel["filled"].sum()),
            "clicks": int(season3_panel["clicked"].sum()),
            "conversions": int(season3_panel["converted"].sum()),
            "fill_rate": season3_panel["filled"].mean(),
            "click_rate_per_opportunity": season3_panel["clicked"].mean(),
            "conversion_rate_per_opportunity": season3_panel["converted"].mean(),
        }
    ]
)
season3_summary

[  2279.1s] Starting: season-three holdout panel construction
[  2279.1s] Building training3rd panel for date 20131019 (all rows).
[  2292.5s] Built training3rd date 20131019: 352,766 opportunities, 227,827 fills, 83 clicks in 13.36s.
[  2292.5s] Building training3rd panel for date 20131020 (all rows).
[  2302.7s] Built training3rd date 20131020: 326,830 opportunities, 214,022 fills, 65 clicks in 10.22s.
[  2302.7s] Building training3rd panel for date 20131021 (all rows).
[  2350.8s] Built training3rd date 20131021: 1,547,653 opportunities, 844,982 fills, 500 clicks in 48.11s.
[  2350.8s] Building training3rd panel for date 20131022 (all rows).
[  2388.7s] Built training3rd date 20131022: 1,212,083 opportunities, 678,805 fills, 379 clicks in 37.84s.
[  2388.7s] Building training3rd panel for date 20131023 (all rows).
[  2426.5s] Built training3rd date 20131023: 1,578,272 opportunities, 225,696 fills, 590 clicks in 37.78s.
[  2426.5s] Building training3rd panel for date 20131024 (all ro

,rows,dates,filled_impressions,clicks,conversions,fill_rate,click_rate_per_opportunity,conversion_rate_per_opportunity
0,10566743,9,3132311,2691,526,0.296431,0.000255,0.00005


## Setup Complete

The workspace now contains the raw-data inventory, the season-two opportunity panel, the season-three holdout panel, price-landscape summaries, and the frozen policy catalog. The next notebooks can use these artifacts to run replay, guardrails, nuisance models, OPE, season-three validation, theory sensitivity, and final DSS decision analysis.